# Rerankers (cross-encoders)

A refresher on the **second stage** of modern retrieval: take the cheap-but-fuzzy candidates a
vector search returns, then re-score each *query–document pair jointly* with a cross-encoder to
get a sharp, relevance-true ordering before you feed the top few to an LLM.

**Domain:** LLM Inference, Training & Optimization  ·  *recommended addition*  ·  **runnable:** yes

## 1. What & Why

A **reranker** is a model that takes a `(query, document)` pair and outputs a single relevance
score for *that specific pair*. You run it over a small candidate set and sort by score. In
practice the reranker is a **cross-encoder**: the query and the document are concatenated and fed
through one transformer together, so every query token can attend to every document token.

The problem it solves: **first-stage retrieval is lossy.** A bi-encoder (the embedding model
behind a vector DB) compresses each document into a *single fixed vector ahead of time*, with no
knowledge of the query, and compresses the query into another vector at search time. Relevance is
then just cosine similarity between two independently-built vectors. That's fast — you precompute
every document vector once and do an ANN lookup — but the compression throws away detail, so the
top-k it returns is approximately right, not exactly right. The true best answer is usually *in*
the top-k but rarely *at the top*.

A cross-encoder never compresses to a vector. It reads the query and document **together** and can
model fine-grained interaction — term overlap, negation, ordering, "does this passage actually
answer this question." That makes it markedly more accurate at ranking. The catch: it must run a
full forward pass for **every candidate at query time** (nothing can be precomputed, because the
score depends on the query), so it's orders of magnitude too slow to run over a whole corpus.

**Reach for a reranker when** retrieval quality is your bottleneck — the right document is being
retrieved but buried at rank 5–20, hurting answer quality or RAG faithfulness. **Don't** reach for
it when first-stage recall is already the problem (the answer isn't in the candidate set at all —
a reranker can only reorder what it's given), or when you can't afford the extra ~10–100 ms per
query and the quality is already good enough.

## 2. Mental Model

**Bi-encoder = sorting résumés by keyword count; cross-encoder = the hiring manager reading each
shortlisted résumé next to the job description.**

The keyword filter is instant and scans thousands of résumés, but it's shallow — it can't tell a
genuine match from one that merely repeats the right words. So you use it to get a *shortlist*,
then a careful reader compares each finalist directly against the role. You'd never have the
manager read all 10,000 résumés (too slow); you'd never hire straight off the keyword count (too
crude). Retrieval pipelines do exactly this two-stage split:

```
            cheap & wide                         expensive & sharp
  query ──► bi-encoder / BM25 ──► top-50 ──► cross-encoder reranker ──► top-5 ──► LLM
            (precomputed vectors,            (one fresh forward pass
             whole corpus, ANN)               per candidate, query-aware)
```

The bi-encoder embeds query and doc **separately** (two towers that never meet → vectors can be
cached). The cross-encoder embeds them **together** (one tower, full attention across the pair →
nothing cacheable, but it sees everything). Wide-and-cheap to find candidates; narrow-and-sharp to
order them.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Bi-encoder** | Two-tower model that encodes query and document *independently* into vectors; relevance = cosine/dot of the two. Document vectors are precomputed → fast ANN search over the whole corpus. This is your first stage. |
| **Cross-encoder** | Single-tower model that takes `[query] [SEP] [document]` together and outputs one relevance score. Query-aware, far more accurate, but **nothing is precomputable** — N candidates ⇒ N forward passes. This is the reranker. |
| **Two-stage retrieval** | Retrieve a wide candidate set cheaply (bi-encoder/BM25), then rerank a small slice (cross-encoder). The standard production pattern. |
| **Top-k (rerank depth)** | How many first-stage candidates you feed the reranker. Bigger k = higher chance the true answer is in the set, but linearly more cost. Typical: retrieve 50–200, rerank to 3–10. |
| **Relevance score** | The cross-encoder's raw output (a logit). Use it to **sort**; don't read it as a calibrated probability unless the model was trained for that. Scores aren't comparable across different rerankers. |
| **MS MARCO** | The passage-ranking dataset most off-the-shelf rerankers are trained on (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2`). Good default for English web/QA text. |
| **LLM / listwise reranker** | Use an LLM to reorder candidates (pointwise score, or rank a whole list in one prompt). More flexible and zero-setup, but slower and pricier than a dedicated cross-encoder. |
| **ColBERT (late interaction)** | A middle ground: token-level embeddings with a cheap max-sim interaction — more precise than a bi-encoder, more scalable than a full cross-encoder. |

## 4. Setup

The first worked example is a **self-contained NumPy toy** — no model download, no network — that
reproduces *why* a reranker helps. The second example shows the real API
(`sentence-transformers`' `CrossEncoder`) and is gated behind an import check **and** an env flag,
so the notebook runs top-to-bottom either way.

```bash
pip install sentence-transformers   # pulls torch + transformers; needed only for Example 2
```

To actually run the real model in Example 2, install the package and set
`RUN_RERANKER_DOWNLOAD=1` (it fetches a ~80 MB checkpoint on first use).

In [1]:
import os
import re
import hashlib
import numpy as np
from collections import Counter

print("numpy", np.__version__)

try:
    import sentence_transformers as st
    print("sentence-transformers", st.__version__, "(Example 2 can run a real model)")
except ImportError:
    print("sentence-transformers not installed (Example 2 falls back to a no-op explanation)")

print("RUN_RERANKER_DOWNLOAD set:", bool(os.getenv("RUN_RERANKER_DOWNLOAD")))

numpy 2.5.0
sentence-transformers not installed (Example 2 falls back to a no-op explanation)
RUN_RERANKER_DOWNLOAD set: False


## 5. Worked Examples

### Example 1 — Why retrieve-then-rerank beats retrieve-alone (pure NumPy)

We model the two stages honestly:

- **First stage (bi-encoder stand-in):** compress every document into a *tiny fixed-size vector*
  via feature hashing, then rank by cosine. The compression is lossy — exactly the property of a
  real embedding model — so hash collisions blur topical meaning and the cheap ranking comes out
  *approximately* right.
- **Second stage (cross-encoder stand-in):** for each candidate, score the `(query, document)`
  pair at **full resolution**, rewarding coverage of *all* query terms and their *proximity* in
  the text. This is query-aware and can't be precomputed — just like a real cross-encoder.

The point isn't the specific scoring functions; it's the **shape**: a lossy, query-blind first
stage retrieves the right answer but buries it, and a sharp, query-aware second stage pulls it back
to the top.

In [2]:
corpus = [
    "Password tips: a good password is long; change your password often and never share it.",
    "Account settings let you update your account name, your email, and your profile photo.",
    "To reset a forgotten password, verify your account by email and then choose a new one.",  # the true answer
    "Factory reset erases the phone; you will reset every app and lose data without a backup.",
    "Two-factor login sends a one-time code after you type your password at sign-in.",
    "Billing handles a failed charge once your bank approves the payment on the account.",
]
query = "reset account password"
TRUE_ANSWER = 2

def toks(s):
    return re.findall(r"[a-z0-9]+", s.lower())

# --- First stage: bi-encoder stand-in = lossy hash embedding + cosine ---
DIM, SEED = 8, 9  # small DIM => collisions => lossy compression, like a real embedding

def stable_hash(token):
    return int(hashlib.md5(f"{SEED}:{token}".encode()).hexdigest(), 16)

def embed(text):
    v = np.zeros(DIM)
    for tok, n in Counter(toks(text)).items():
        v[stable_hash(tok) % DIM] += n          # different tokens can collide into one slot
    norm = np.linalg.norm(v)
    return v / norm if norm else v

doc_vecs = np.stack([embed(d) for d in corpus])  # precomputed once, query-blind
q_vec = embed(query)
cosine = doc_vecs @ q_vec
first_stage = list(np.argsort(-cosine))

print("FIRST STAGE — bi-encoder cosine (cheap, query-blind):")
for rank, i in enumerate(first_stage, 1):
    flag = "  <- true answer" if i == TRUE_ANSWER else ""
    print(f"  {rank}. doc{i}  cos={cosine[i]:.3f}  {corpus[i][:48]}{flag}")

FIRST STAGE — bi-encoder cosine (cheap, query-blind):
  1. doc3  cos=0.653  Factory reset erases the phone; you will reset e
  2. doc0  cos=0.600  Password tips: a good password is long; change y
  3. doc2  cos=0.345  To reset a forgotten password, verify your accou  <- true answer
  4. doc4  cos=0.345  Two-factor login sends a one-time code after you
  5. doc5  cos=0.141  Billing handles a failed charge once your bank a
  6. doc1  cos=0.129  Account settings let you update your account nam


In [3]:
# --- Second stage: cross-encoder stand-in = joint, full-resolution pair scoring ---
q_terms = toks(query)

def cross_encode(query_terms, document):
    """Score a (query, document) PAIR together: coverage of all query terms + their proximity."""
    words = toks(document)
    positions = {}
    for idx, w in enumerate(words):
        positions.setdefault(w, []).append(idx)
    present = [t for t in query_terms if t in positions]
    coverage = len(present) / len(query_terms)
    proximity = 0.0
    if len(present) >= 2:                       # how tightly the matched terms cluster
        firsts = sorted(positions[t][0] for t in present)
        proximity = (len(present) - 1) / (firsts[-1] - firsts[0])
    return coverage + 0.5 * proximity

RERANK_DEPTH = 4
candidates = first_stage[:RERANK_DEPTH]          # only rerank the cheap top-k
scores = {i: cross_encode(q_terms, corpus[i]) for i in candidates}
reranked = sorted(candidates, key=lambda i: -scores[i])

print(f"SECOND STAGE — cross-encoder rerank of the top-{RERANK_DEPTH} (sharp, query-aware):")
for rank, i in enumerate(reranked, 1):
    flag = "  <- true answer" if i == TRUE_ANSWER else ""
    print(f"  {rank}. doc{i}  score={scores[i]:.3f}  {corpus[i][:48]}{flag}")

bi_rank = first_stage.index(TRUE_ANSWER) + 1
ce_rank = reranked.index(TRUE_ANSWER) + 1
print(f"\nTrue answer (doc{TRUE_ANSWER}): bi-encoder put it at rank {bi_rank} -> reranked to rank {ce_rank}.")
print(f"Cross-encoder forward passes this query: {len(candidates)} "
      f"(one per candidate) vs {len(corpus)} docs embedded once for the whole corpus.")

SECOND STAGE — cross-encoder rerank of the top-4 (sharp, query-aware):
  1. doc2  score=1.167  To reset a forgotten password, verify your accou  <- true answer
  2. doc3  score=0.333  Factory reset erases the phone; you will reset e
  3. doc0  score=0.333  Password tips: a good password is long; change y
  4. doc4  score=0.333  Two-factor login sends a one-time code after you

True answer (doc2): bi-encoder put it at rank 3 -> reranked to rank 1.
Cross-encoder forward passes this query: 4 (one per candidate) vs 6 docs embedded once for the whole corpus.


**What just happened:** the lossy first stage retrieved the correct passage but ranked a
topically-similar distractor above it. Reranking the top-4 by joint coverage+proximity promoted the
true answer to rank 1 — and it only cost one forward pass per *candidate*, not per *document*. That
last line is the whole economic argument: you pay the expensive model only on a tiny shortlist.

### Example 2 — A real cross-encoder (`sentence-transformers`), gated

This is the production-shaped call. `CrossEncoder.predict` takes a list of `[query, document]`
pairs and returns one score per pair — exactly the second stage above, but with a transformer
trained on MS MARCO instead of our toy function. It's gated behind an import check **and**
`RUN_RERANKER_DOWNLOAD` (the model is an ~80 MB download), so the cell always executes; set the env
var to run it for real.

In [4]:
def rerank_with_cross_encoder(query, documents, model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"):
    """Real two-stage second pass: score every (query, doc) pair and sort high-to-low."""
    from sentence_transformers import CrossEncoder
    model = CrossEncoder(model_name)
    pairs = [[query, doc] for doc in documents]      # the join the cross-encoder reads together
    scores = model.predict(pairs)                    # one forward pass per pair
    order = np.argsort(-scores)
    return [(int(i), float(scores[i])) for i in order]

if os.getenv("RUN_RERANKER_DOWNLOAD") and "st" in dir():
    ranked = rerank_with_cross_encoder(query, corpus)
    print(f"Real cross-encoder ranking for: {query!r}")
    for rank, (i, score) in enumerate(ranked, 1):
        flag = "  <- true answer" if i == TRUE_ANSWER else ""
        print(f"  {rank}. doc{i}  score={score:+.3f}  {corpus[i][:48]}{flag}")
else:
    print("Skipping the real model (set RUN_RERANKER_DOWNLOAD=1 and pip install sentence-transformers).")
    print("The call shape is:")
    print('    model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")')
    print('    scores = model.predict([[query, doc] for doc in candidates])')
    print("    # then: sort candidates by score, keep the top few, hand them to the LLM")

Skipping the real model (set RUN_RERANKER_DOWNLOAD=1 and pip install sentence-transformers).
The call shape is:
    model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    scores = model.predict([[query, doc] for doc in candidates])
    # then: sort candidates by score, keep the top few, hand them to the LLM


## 6. Gotchas & Pitfalls

- **A reranker can't fix bad recall.** It only reorders the candidate set it's handed. If the
  answer isn't in the first-stage top-k, no reranker will surface it — widen retrieval (bigger k,
  hybrid BM25+vector, better embeddings) *before* reaching for a reranker.
- **Rerank depth is a real cost knob.** Cost scales linearly with the number of candidates because
  each is a separate forward pass. Reranking 500 candidates per query will wreck your latency and
  bill. Retrieve wide (say 100–200), rerank a slice (say 50), keep the top 3–10.
- **Scores are for sorting, not thresholding (usually).** Off-the-shelf cross-encoder outputs are
  raw logits, not calibrated probabilities, and they're **not comparable across models** or even
  across queries. If you need a "is this relevant at all?" cutoff, calibrate on your own data.
- **Watch the token limit and truncation.** The pair `query + document` shares one context window
  (often 512 tokens). Long documents get silently truncated — the part holding the answer may be
  cut. Chunk documents to a size that fits, or use a long-context reranker.
- **Latency is per-query and unavoidable.** Unlike bi-encoder vectors, nothing can be precomputed —
  the score depends on the live query. Budget for it: batch the pairs, use a small model
  (MiniLM-L-6 over L-12), run on GPU if throughput matters, or cache results for repeated queries.
- **Use the right model for your domain/language.** MS MARCO rerankers are English web/QA. For
  multilingual, code, or specialized domains, pick a matching checkpoint (e.g. BGE / mmarco /
  domain-tuned) or fine-tune — a mismatched reranker can *hurt* ordering.
- **Don't double-encode mismatched text.** Feed the reranker the same chunk text your bi-encoder
  indexed. Reranking against a different granularity (full doc vs. the chunk that was retrieved)
  produces confusing scores.

## 7. When to Use vs Alternatives

| Approach | Best when | Cost / downside |
|---|---|---|
| **Bi-encoder only** (vector search) | Latency-critical, huge corpus, "good enough" ordering; the only thing that scales to millions of docs as the *first* stage. | Lossy, query-blind ranking; the true best result is often buried below rank 1. |
| **Cross-encoder reranker** | You already have decent recall and need the *ordering* sharpened — RAG faithfulness, top-3 quality, search relevance. The standard second stage. | One forward pass per candidate at query time; nothing precomputable; adds ~10–100 ms. |
| **ColBERT / late interaction** | You want better-than-bi-encoder precision but can't afford a full cross-encoder per candidate; token-level match at scale. | Bigger index (per-token vectors), more infra complexity than plain ANN. |
| **LLM reranker (pointwise or listwise)** | Zero model setup, you already call an LLM, need flexible/instructable relevance (or few candidates). | Slower and pricier per query than a dedicated cross-encoder; output ordering can be unstable. |
| **Hybrid retrieval (BM25 + vectors)** | The failure is *recall*, not ordering — rare terms, exact IDs, keywords embeddings miss. | Fixes what gets retrieved, not how it's ranked; often paired *with*, not instead of, a reranker. |

**Rule of thumb:** fix recall first (hybrid retrieval, better embeddings, bigger k), then add a
cross-encoder reranker to fix ordering. They're complements: retrieve wide and cheap, rerank narrow
and sharp.

## 8. Resources

- **Sentence-Transformers — Cross-Encoder docs & usage**: https://www.sbert.net/examples/applications/cross-encoder/README.html
- **Sentence-Transformers — Retrieve & Re-Rank pattern (the two-stage pipeline)**: https://www.sbert.net/examples/applications/retrieve_rerank/README.html
- **Nogueira & Cho, 2019 — "Passage Re-ranking with BERT"** (the original BERT cross-encoder reranker): https://arxiv.org/abs/1901.04085
- **Khattab & Zaharia, 2020 — "ColBERT: Efficient Passage Search via Late Interaction"**: https://arxiv.org/abs/2004.12832
- **Pinecone — "Rerankers and Two-Stage Retrieval" (practical explainer)**: https://www.pinecone.io/learn/series/rag/rerankers/
- **Cohere Rerank — a hosted reranker API (call shape & when to use)**: https://docs.cohere.com/docs/rerank-overview

Related notebooks in this domain: [`semantic-search`](semantic-search.ipynb),
[`vector-embeddings`](vector-embeddings.ipynb), and [`rag`](rag.ipynb) — the reranker is the second
stage that sits on top of all three.